# mySports Video Creator 🏒

Dieses Notebook schneidet die mySports-Reels im freigegebenen Design (9:16, Figtree Bold, Sunrise Red).
Texte, Untertitel und Specs kommen vom **Gemini-Gem «mySports Video Creator»**, hier wird nur gerendert.

**So gehts:** Zellen von oben nach unten mit ▶ ausführen. Schritt 1 immer zuerst.
Die Videos liegen im Google Drive im Projektordner, nie auf GitHub.

## 1 · Einrichten (jedes Mal zuerst)

In [ ]:
#@title Projektordner auf Google Drive { display-mode: "form" }
ORDNER = "mySports - SOM Organic - Video Creator"  #@param {type:"string"}
REPO = "https://github.com/dan-b-h/mysports-video-creator.git"

from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob
# Ordner in «Meine Ablage» und in allen geteilten Ablagen suchen
treffer = [p for p in glob.glob('/content/drive/MyDrive/**/' + glob.escape(ORDNER), recursive=True)
           + glob.glob('/content/drive/Shareddrives/*/**/' + glob.escape(ORDNER), recursive=True) if os.path.isdir(p)]
assert treffer, f"Ordner «{ORDNER}» nicht gefunden. Ist er für dich freigegeben bzw. in «Meine Ablage» verknüpft?"
PROJEKT = treffer[0]
print('Projektordner:', PROJEKT)
if os.path.isdir('/content/mvc'):
    subprocess.run(['git', '-C', '/content/mvc', 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', REPO, '/content/mvc'], check=True)
# Brand-Dateien (Schriften, Logo) liegen auf Drive und werden für die Skripte verlinkt
if not os.path.exists('/content/mvc/03_Brand'):
    os.symlink(os.path.join(PROJEKT, '03_Brand'), '/content/mvc/03_Brand')
os.chdir(PROJEKT)
V = '/content/mvc/vorlage'
print('Bereit. Arbeitsordner:', os.getcwd())
print('Originalvideos:', len(os.listdir('Originalvideos')), 'Dateien')

Mounted at /content/drive


## 2 · Video sichten (optional)
Erstellt Kontaktbögen mit Zeitstempeln, damit du Tore und Szenen schnell findest.
Die Bilder landen in `04_Arbeitsdateien/<Datum>/check/`.

In [ ]:
#@title Kontaktbogen { display-mode: "form" }
VIDEO = "Originalvideos/261004_HCA_HCL_DE.mp4"  #@param {type:"string"}
DATUM = "261004"  #@param {type:"string"}
import glob
from IPython.display import Image, display
out = f"04_Arbeitsdateien/{DATUM}/check"; os.makedirs(out, exist_ok=True)
name = os.path.splitext(os.path.basename(VIDEO))[0]
font = os.path.join(PROJEKT, '03_Brand', 'Figtree-Bold.ttf')
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-skip_frame', 'nokey', '-i', VIDEO, '-vf',
    f"scale=320:-1,drawtext=fontfile='{font}':text='%{{pts\\:hms}}':x=4:y=4:fontsize=20:fontcolor=yellow:box=1:boxcolor=black,tile=6x6",
    '-vsync', 'vfr', f'{out}/{name}_%02d.png'], check=True)
for p in sorted(glob.glob(f'{out}/{name}_*.png')): display(Image(p, width=1100))

## 3 · Interview transkribieren (nur für Interview- und Kombi-Reels)
Erstellt ein Transkript mit Zeitangaben. Das Transkript gibst du dem Gem; er liefert dir die bereinigten Untertitel zurück.
Beim ersten Mal werden die Spracherkennungs-Modelle geladen (ca. 2 Minuten), danach liegen sie im Drive.

In [ ]:
#@title Transkript { display-mode: "form" }
VIDEO = "Originalvideos/261003_SCRJ_SCB_INTERVIEWS_DE.mp4"  #@param {type:"string"}
DATUM = "261003"  #@param {type:"string"}
SPRACHE = "de"  #@param ["de", "fr", "en", "it"]
subprocess.run('pip -q install sherpa-onnx', shell=True, check=True)
MOD = os.path.join(PROJEKT, '_modelle')
os.makedirs(MOD, exist_ok=True)
if not os.path.exists(f'{MOD}/sherpa-onnx-whisper-turbo/turbo-encoder.int8.onnx'):
    subprocess.run(f'cd "{MOD}" && curl -sSL https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-whisper-turbo.tar.bz2 | tar xj', shell=True, check=True)
if not os.path.exists(f'{MOD}/silero_vad.onnx'):
    subprocess.run(f'curl -sSL -o "{MOD}/silero_vad.onnx" https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/silero_vad.onnx', shell=True, check=True)
os.makedirs(f'04_Arbeitsdateien/{DATUM}/audio', exist_ok=True)
WAV = f"04_Arbeitsdateien/{DATUM}/audio/{os.path.splitext(os.path.basename(VIDEO))[0]}.wav"
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', VIDEO, '-vn', '-ac', '1', '-ar', '16000', WAV], check=True)
env = dict(os.environ, MODELLE=MOD, SPRACHE=SPRACHE)
r = subprocess.run(['python3', f'{V}/audio_tools.py', 'transcribe', WAV], capture_output=True, text=True, env=env)
txt = '\n'.join(l for l in r.stdout.splitlines() if not l.startswith('/project'))
open(WAV.replace('.wav', '_Transkript.txt'), 'w').write(txt)
print(txt or r.stderr)
print('\nSprechpausen (für saubere Schnitte):')
print(subprocess.run(['python3', f'{V}/audio_tools.py', 'pauses', WAV], capture_output=True, text=True, env=env).stdout)

## 4 · Spec speichern
Füge hier die Spec (JSON) ein, die dir der Gem geliefert hat.
Bei Interview- und Kombi-Reels werden die Wortzeiten der Untertitel automatisch berechnet.

In [ ]:
#@title Spec { display-mode: "form" }
DATUM = "261004"  #@param {type:"string"}
NAME = "hca_split_v1"  #@param {type:"string"}
SPEC = r"""
{ "hier": "JSON vom Gem einfügen" }
"""
import json
spec = json.loads(SPEC)
pfad = f"04_Arbeitsdateien/{DATUM}/{NAME}.json"; os.makedirs(os.path.dirname(pfad), exist_ok=True)

def wortzeiten(teil, quelle):
    """Untertitel mit src_s/src_e -> Clip-Zeiten und Wort-für-Wort-Einblendung."""
    if not teil.get('cues') or 'word_t' in teil['cues'][0]: return teil
    wav = f"04_Arbeitsdateien/{DATUM}/audio/{os.path.splitext(os.path.basename(quelle))[0]}.wav"
    if not os.path.exists(wav):
        os.makedirs(os.path.dirname(wav), exist_ok=True)
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', quelle, '-vn', '-ac', '1', '-ar', '16000', wav], check=True)
    subprocess.run('pip -q install sherpa-onnx', shell=True, check=True)
    tmp_in, tmp_out = '/content/cues_in.json', '/content/cues_out.json'
    json.dump(teil, open(tmp_in, 'w'), ensure_ascii=False)
    r = subprocess.run(['python3', f'{V}/audio_tools.py', 'wordtimes', wav, tmp_in, tmp_out], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    return json.load(open(tmp_out))

if 'interview' in spec:            # Kombi
    spec['interview'] = wortzeiten(spec['interview'], spec['interview']['source'])
elif 'cues' in spec:               # Interview
    spec = wortzeiten(spec, spec['source'])
json.dump(spec, open(pfad, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Gespeichert:', pfad)

## 5 · Rendern
Wähle das Format und die Spec. Das fertige Video landet in `05_Ausgabe/<Datum>/` und wird unten angezeigt.
**Rechte:** Pro Quelldatei höchstens 30 Sekunden. Längere Specs brechen mit einer Fehlermeldung ab (Ausnahme nur mit `"allow_over_30": true` und Freigabe).

In [ ]:
#@title Video rendern { display-mode: "form" }
FORMAT = "Tore-Splitscreen"  #@param ["Tore-Splitscreen", "Interview", "Kombi Tore + Interview"]
DATUM = "261004"  #@param {type:"string"}
NAME = "hca_split_v1"  #@param {type:"string"}
AUSGABE = "261004_DE_HCA_HCL_Tore_Split_v1"  #@param {type:"string"}
skript = {"Tore-Splitscreen": "render_goals_split.py", "Interview": "render_interview.py",
          "Kombi Tore + Interview": "render_combo.py"}[FORMAT]
ordner = {"Tore-Splitscreen": "Spielhighlights", "Interview": "Interview", "Kombi Tore + Interview": "Interview"}[FORMAT]
ziel = f"05_Ausgabe/{DATUM}/{ordner}/{AUSGABE}.mp4"; os.makedirs(os.path.dirname(ziel), exist_ok=True)
r = subprocess.run(['python3', f'{V}/{skript}', f'04_Arbeitsdateien/{DATUM}/{NAME}.json', ziel], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode == 0:
    from IPython.display import HTML
    from base64 import b64encode
    klein = '/content/vorschau.mp4'
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', ziel, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '28', '-c:a', 'aac', klein], check=True)
    display(HTML(f'<video width=360 controls src="data:video/mp4;base64,{b64encode(open(klein,"rb").read()).decode()}"></video>'))
    print('Fertig:', ziel)

## 6 · Kontrolle vor dem Versand
- Spielstand springt genau beim Tor um, Torschütze und Minute stimmen
- Gesicht bzw. Torszene ist gut im Bild, nichts Wichtiges abgeschnitten
- Untertitel: jedes Wort korrekt, keine Sätze abgehackt
- Anfang und Schluss sind sauber (kein halbes Wort)
- Nutzungsprotokoll in `02_Daten/nutzungsprotokoll.csv` nachgeführt (Quelldatei, Start, Ende, Summe)
- Freigabe der Kundin eingeholt